# Step 3: Dataset Formatting & LLM Fine-Tuning

This notebook demonstrates the process of building the dataset using a teacher LLM (Gemini) and then fine-tuning a small on-device LLM (like Qwen or SmolLM).

## 0a. Google Colab Setup (Step 1)
Run this cell FIRST to install Conda in Google Colab. 
**Note:** Colab will automatically restart the kernel after this cell finishes. This is normal! Wait for it to reconnect before moving to Step 1b.

In [ ]:
try:
    import google.colab
    !pip install -q condacolab
    import condacolab
    condacolab.install()
except ImportError:
    print("Not running in Colab. Skipping Conda setup.")

## 0b. Mount Drive & Load Environment (Step 2)
After the kernel restarts, run this cell to mount your Google Drive, navigate to the project folder, and install all dependencies from `environment.yml`.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    %cd "/content/drive/MyDrive/04 - University/04.02 - Adelaide University/04.02.02 - Sem 1 2026/02 - Deep Learning Applications/0 - Final Project/call-contextual-extractor"
    
    print("\n--- Installing Environment ---")
    !conda env update -n base -f environment.yml
    
    print("\n--- Installing Colab Unsloth Drivers ---")
    !pip install pandas
    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install --no-deps "xformers<0.0.27" peft accelerate bitsandbytes
except ImportError:
    print("Not running in Colab. Skipping mount and environment update.")

## 1. Dataset Generation
Run the `dataset_builder.py` script. It will scan all `final_dialogue.json` files and use Gemini to extract structured CRM fields. It includes checkpointing, so you can stop and resume anytime.

In [2]:
!python pipeline/dataset_builder.py

Found 400 dialogue files. 25 already processed.
Building Dataset: 100%|███████████████████████| 400/400 [32:35<00:00,  4.89s/it]
Dataset building complete. Added 375 new records to data/finetuning_dataset.jsonl


### Preview the generated dataset

In [3]:
import json
import pandas as pd

try:
    with open('data/finetuning_dataset.jsonl', 'r', encoding='utf-8') as f:
        # Load first 5 records
        data = [json.loads(next(f)) for _ in range(5)]
    df = pd.DataFrame(data)
    display(df)
except Exception as e:
    print("Run the dataset builder first to generate the dataset.", e)

,instruction,input,response
0,You are a structured-field extractor for teles...,Agent: à dạ a lô ạ à chị ạ em là hòa trương vi...,"{""customer_sector"": ""Bất động sản"", ""customer_..."
1,You are a structured-field extractor for teles...,Agent: a lô cho em hỏi đây cái số chị của anh\...,"{""customer_sector"": null, ""customer_needs"": ""i..."
2,You are a structured-field extractor for teles...,Customer: a lô\nAgent: vẻ luôn ạ dạ em chào ch...,"{""customer_sector"": null, ""customer_needs"": ""T..."
3,You are a structured-field extractor for teles...,Customer: a lô\nAgent: dạ em chào anh ạ anh có...,"{""customer_sector"": ""Retail and Dental clinic""..."
4,You are a structured-field extractor for teles...,Customer: a lô\nAgent: a lô à dạ vâng em chào ...,"{""customer_sector"": ""Bất động sản"", ""customer_..."


## 2. Model Fine-Tuning
Now we fine-tune a small LLM using LoRA via the `unsloth` library. We are running consecutively over:
- `Qwen/Qwen3.5-2B`
- `Qwen/Qwen3.5-0.8B`
- `google/gemma-4-E4B`

In [ ]:
!python pipeline/fine_tuner.py --model Qwen/Qwen3.5-2B --save_name Qwen3.5-2B
!python pipeline/fine_tuner.py --model Qwen/Qwen3.5-0.8B --save_name Qwen3.5-0.8B
!python pipeline/fine_tuner.py --model google/gemma-4-E4B --save_name gemma-4-E4B

## 3. Inference / Evaluation
Once the model is fine-tuned, you can load the generated LoRA weights to run inference on new transcripts.

In [ ]:
!python pipeline/inference.py --model data/models/Qwen3.5-0.8B_lora